# 🧪 생성 품질 상세 지표 분석 보고서

이 보고서는 로컬 LLM 환경(Scenario A)의 프롬프트 개선 과정과 클라우드 API 환경(Scenario B)의 최종 최적화 성능을 비교 분석합니다.

## 📊 1. 실험군별 상세 설정 및 품질 지표 비교

| 실험명 | 인프라 | LLM 모델 | 임베딩 모델 | 주요 전략 | Faithfulness | Correctness | Groundedness | Completeness | Citation |
| :--- | :--- | :--- | :--- | :--- | :---: | :---: | :---: | :---: | :---: |
| **prompts-exp01** | 로컬 (A) | Gemma-4-E4B-it | BAAI/bge-m3 | Baseline | 0.80 | 0.50 | 0.78 | 0.60 | 0.85 |
| **prompts-P4_C2_F5_S3_T5** | 로컬 (A) | Gemma-4-E4B-it | BAAI/bge-m3 | **P4 프롬프트** | **0.95** | **0.65** | **0.94** | **0.85** | **0.98** |
| **ScenarioB-Optimized** | 클라우드 (B) | GPT-5-mini | OpenAI-3-small | **Anchor/RRF-Opt** | **1.00** | **0.80** | **1.00** | **0.95** | **1.00** |

---

## 🔍 2. 실험 환경 및 설정 스냅샷 (Configuration Snapshot)

### 1️⃣ 시나리오 A (prompts-exp01 / P4_C2_F5_S3_T5)
- **인프라:** 시나리오 A 환경 (Local GPU 가속)
- **LLM 모델:** `google/gemma-4-E4B-it` (128K context, 4-bit quantization)
- **임베딩:** `BAAI/bge-m3` (HuggingFace Local)
- **주요 전략:** 
  - **Baseline:** 기본 시스템 프롬프트 사용
  - **P4:** English CoT(Chain of Thought) 사고 과정 강제 + 구조화된 한국어 출력 형식(핵심 답변/근거/누락 항목) 도입
- **공통 설정:** Chunk Size 1000, Overlap 150, Hybrid Search(RRF K=60)

### 2️⃣ 시나리오 B (ScenarioB-Optimized)
- **인프라:** 시나리오 B 환경 (OpenAI API 기반 Cloud)
- **LLM 모델:** `gpt-5-mini` 
- **임베딩:** `text-embedding-3-small` (OpenAI)
- **주요 전략:**
  - **`anchor_auxiliary: true`**: 고유 명사 및 수치 데이터 검색 정확도 보강
  - **`pool_multiplier: 5`**: 검색 후보군을 5배수로 확대하여 정답 포함 확률 극대화
  - **`llm_with_rule_fallback`**: LLM 기반 지능형 질의 재작성을 통한 멀티턴 맥락 파악

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 시각화 설정
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'NanumGothic'

data = [
    {"Exp": "exp01", "Faithfulness": 0.80, "Correctness": 0.50, "Groundedness": 0.78, "Completeness": 0.60, "Citation": 0.85},
    {"Exp": "P4-Opt", "Faithfulness": 0.95, "Correctness": 0.65, "Groundedness": 0.94, "Completeness": 0.85, "Citation": 0.98},
    {"Exp": "ScenarioB", "Faithfulness": 1.00, "Correctness": 0.80, "Groundedness": 1.00, "Completeness": 0.95, "Citation": 1.00}
]

df = pd.DataFrame(data).melt(id_vars="Exp", var_name="Metric", value_name="Score")

plt.figure(figsize=(14, 7))
sns.barplot(data=df, x="Metric", y="Score", hue="Exp", palette="viridis")
plt.title("Comparison of Generation Quality Metrics", fontsize=16)
plt.ylim(0, 1.1)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()